<a href="https://colab.research.google.com/github/nicopicomoco-sekihan-80/ShortthoughtLM/blob/main/V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install "transformers==4.57.3" "accelerate>=1.10,<2" "datasets" "sentencepiece"


In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-1.7B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()

print("Loaded:", MODEL_NAME)
print("Device:", next(model.parameters()).device)


PyTorch: 2.11.0+cu130
CUDA: 13.0
Transformers: 4.57.3
Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/622M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-1.7B
Device: cuda:0


In [ ]:
# ============================================
# Qwen3 1.7B - GSM8K Baseline
# Normal vs Necessary-Only Thinking
# ============================================

from datasets import load_dataset
import re
import time
import torch

# ============================================
# Config
# ============================================

N_SAMPLES = 10
MAX_NEW_TOKENS = 2048

MODEL_NAME = "Qwen/Qwen3-1.7B"

print("Model:", MODEL_NAME)
print("Samples:", N_SAMPLES)
print("Max new tokens:", MAX_NEW_TOKENS)
print("Device:", next(model.parameters()).device)


# ============================================
# Load GSM8K
# ============================================

print("\nLoading GSM8K...")

dataset = load_dataset(
    "openai/gsm8k",
    "main",
    split="test"
)

print("Dataset:", len(dataset))


# ============================================
# Prompts
# ============================================

PROMPTS = {

    "normal": (
        "Solve the math problem carefully. "
        "Give the correct final answer."
    ),

    "necessary_only": (
        "Solve the math problem carefully. "
        "Use only the reasoning necessary to arrive at the correct answer. "
        "Avoid unnecessary or repetitive thinking. "
        "Do not spend tokens on reasoning that does not help solve the problem. "
        "Give the final answer clearly."
    ),
}


# ============================================
# Utilities
# ============================================

def extract_gold(answer):

    match = re.search(
        r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        answer
    )

    if not match:
        return None

    return match.group(1).replace(",", "")


def extract_prediction(text):

    # Remove <think>...</think>
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL
    ).strip()

    # Find numbers
    numbers = re.findall(
        r"[-+]?\d+(?:,\d{3})*(?:\.\d+)?",
        text
    )

    if not numbers:
        return None

    return numbers[-1].replace(",", "")


def count_tokens(text):

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )


# ============================================
# Single evaluation
# ============================================

def evaluate_condition(condition_name, instruction):

    print("\n")
    print("=" * 70)
    print(f"CONDITION: {condition_name}")
    print("=" * 70)

    results = []

    for i in range(N_SAMPLES):

        item = dataset[i]

        question = item["question"]
        gold = extract_gold(item["answer"])

        # -------------------------------
        # Chat prompt
        # -------------------------------

        messages = [
            {
                "role": "user",
                "content": (
                    instruction
                    + "\n\n"
                    + question
                )
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to("cuda")

        input_len = inputs["input_ids"].shape[1]

        print(
            f"\n[{i+1:02d}/{N_SAMPLES}] "
            f"input={input_len}"
        )

        # -------------------------------
        # Generate
        # -------------------------------

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        # -------------------------------
        # Decode
        # -------------------------------

        generated_ids = outputs[0][input_len:]

        generated_text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=False
        )

        # -------------------------------
        # Think / final
        # -------------------------------

        think_match = re.search(
            r"<think>(.*?)</think>",
            generated_text,
            flags=re.DOTALL
        )

        if think_match:

            think_text = think_match.group(1).strip()

            final_text = generated_text[
                think_match.end():
            ].strip()

        else:

            think_text = ""
            final_text = generated_text.strip()

        think_tokens = count_tokens(think_text)

        final_tokens = count_tokens(final_text)

        total_tokens = len(generated_ids)

        # -------------------------------
        # Answer
        # -------------------------------

        predicted = extract_prediction(
            final_text
        )

        correct = (
            predicted is not None
            and gold is not None
            and predicted == gold
        )

        hit_limit = (
            total_tokens >= MAX_NEW_TOKENS
        )

        results.append({
            "index": i + 1,
            "condition": condition_name,
            "correct": correct,
            "gold": gold,
            "predicted": predicted,
            "think_tokens": think_tokens,
            "final_tokens": final_tokens,
            "total_tokens": total_tokens,
            "time": elapsed,
            "limit": hit_limit,
        })

        print(
            f"correct={correct} "
            f"think={think_tokens:4d} "
            f"final={final_tokens:4d} "
            f"total={total_tokens:4d} "
            f"time={elapsed:6.2f}s "
            f"limit={hit_limit}"
        )

    return results


# ============================================
# Run both conditions
# ============================================

all_results = []

for condition_name, instruction in PROMPTS.items():

    condition_results = evaluate_condition(
        condition_name,
        instruction
    )

    all_results.extend(
        condition_results
    )


# ============================================
# Summary
# ============================================

print("\n\n")
print("=" * 80)
print("BASELINE SUMMARY")
print("=" * 80)

for condition_name in PROMPTS.keys():

    results = [
        r for r in all_results
        if r["condition"] == condition_name
    ]

    n = len(results)

    accuracy = (
        sum(r["correct"] for r in results) / n
    )

    avg_think = (
        sum(r["think_tokens"] for r in results) / n
    )

    avg_final = (
        sum(r["final_tokens"] for r in results) / n
    )

    avg_total = (
        sum(r["total_tokens"] for r in results) / n
    )

    avg_time = (
        sum(r["time"] for r in results) / n
    )

    limits = sum(
        r["limit"] for r in results
    )

    print(f"\n{condition_name}")
    print("-" * 40)

    print(
        f"Accuracy:        {accuracy:.3f}"
    )

    print(
        f"Correct:         "
        f"{sum(r['correct'] for r in results)}/{n}"
    )

    print(
        f"Avg think:       "
        f"{avg_think:.1f} tokens"
    )

    print(
        f"Avg final:       "
        f"{avg_final:.1f} tokens"
    )

    print(
        f"Avg total:       "
        f"{avg_total:.1f} tokens"
    )

    print(
        f"Avg latency:     "
        f"{avg_time:.2f} sec"
    )

    print(
        f"Hit token limit: "
        f"{limits}/{n}"
    )


# ============================================
# Direct comparison
# ============================================

normal = [
    r for r in all_results
    if r["condition"] == "normal"
]

necessary = [
    r for r in all_results
    if r["condition"] == "necessary_only"
]


def avg(results, key):
    return sum(r[key] for r in results) / len(results)


normal_think = avg(normal, "think_tokens")
necessary_think = avg(necessary, "think_tokens")

normal_total = avg(normal, "total_tokens")
necessary_total = avg(necessary, "total_tokens")

normal_time = avg(normal, "time")
necessary_time = avg(necessary, "time")


print("\n")
print("=" * 80)
print("NORMAL → NECESSARY-ONLY")
print("=" * 80)

print(
    f"\nThink tokens:"
    f" {normal_think:.1f} → {necessary_think:.1f}"
)

print(
    f"Total tokens:"
    f" {normal_total:.1f} → {necessary_total:.1f}"
)

print(
    f"Latency:"
    f" {normal_time:.2f}s → {necessary_time:.2f}s"
)

if normal_think > 0:

    think_reduction = (
        1 - necessary_think / normal_think
    ) * 100

    print(
        f"\nThink reduction:"
        f" {think_reduction:.1f}%"
    )

if normal_total > 0:

    total_reduction = (
        1 - necessary_total / normal_total
    ) * 100

    print(
        f"Total token reduction:"
        f" {total_reduction:.1f}%"
    )

if normal_time > 0:

    time_reduction = (
        1 - necessary_time / normal_time
    ) * 100

    print(
        f"Latency reduction:"
        f" {time_reduction:.1f}%"
    )


Model: Qwen/Qwen3-1.7B
Samples: 10
Max new tokens: 2048
Device: cuda:0

Loading GSM8K...
Dataset: 1319


CONDITION: normal

[01/10] input=86
correct=True think=1409 final= 264 total=1677 time= 79.33s limit=False

[02/10] input=47
correct=True think=1036 final= 183 total=1223 time= 57.22s limit=False

[03/10] input=78
correct=False think=   0 final=2048 total=2048 time= 95.45s limit=True

[04/10] input=56
correct=True think= 963 final= 205 total=1172 time= 55.13s limit=False

[05/10] input=132
correct=True think=1347 final= 261 total=1612 time= 75.89s limit=False

[06/10] input=77
correct=False think=   0 final=2048 total=2048 time= 96.07s limit=True

[07/10] input=63
correct=True think= 817 final= 283 total=1104 time= 52.15s limit=False

[08/10] input=91
correct=False think=   0 final=2048 total=2048 time= 95.37s limit=True

[09/10] input=125
correct=False think=   0 final=2048 total=2048 time= 96.43s limit=True

[10/10] input=81
correct=True think= 922 final= 291 total=1217 time= 57.1

In [ ]:
# ============================================================
# Install
# ============================================================

!pip -q install -U \
    "torchao>=0.16.0" \
    "transformers==4.57.3" \
    "trl>=0.24.0" \
    "peft>=0.17.0" \
    "datasets" \
    "accelerate" \
    "bitsandbytes" \
    "sentencepiece"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 37.8 MB/s eta 0:00:00


In [ ]:
!pip install -U --no-cache-dir bitsandbytes


In [ ]:
!pip uninstall -y bitsandbytes
!pip install --no-cache-dir -U bitsandbytes


Found existing installation: bitsandbytes 0.50.2
Uninstalling bitsandbytes-0.50.2:
  Successfully uninstalled bitsandbytes-0.50.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 216.1 MB/s eta 0:00:00


In [ ]:
import torch
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("bitsandbytes:", bnb.__version__)


PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4
bitsandbytes: 0.50.2


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen3-1.7B"

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# tokenizer
print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# model
print("\nLoading 4bit Qwen3...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False

print("\n==============================")
print("SUCCESS")
print("==============================")
print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)


PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4

Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



Loading 4bit Qwen3...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


SUCCESS
Model: Qwen/Qwen3-1.7B
Device: cuda:0


In [ ]:
print(
    "Allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print(
    "Reserved:",
    round(
        torch.cuda.memory_reserved() / 1024**3,
        2
    ),
    "GB"
)


Allocated: 1.26 GB
Reserved: 2.84 GB


In [ ]:
# ============================================================
# Qwen3-1.7B 4bit QLoRA + GRPO v2
#
# Objective:
#
#   1. Correctness is ALWAYS more important than brevity.
#   2. Among correct answers:
#          shorter reasoni
ng -> higher reward
#   3. Among incorrect answers:
#          same reward
#
# Therefore:
#
#   correct short  > correct long  >>  incorrect
#
# Generation limit is 3000 tokens.
# This is only a safety limit, NOT a training objective.
#
# IMPORTANT:
# Start from a CLEAN 4bit base model.
# Do NOT run this on a model that already has a PEFT adapter.
# ============================================================

import os
import re
import time
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import GRPOConfig, GRPOTrainer


# ============================================================
# 0. Configuration
# ============================================================

MODEL_NAME = "Qwen/Qwen3-1.7B"

OUTPUT_DIR = "./qwen3_1.7b_grpo_qlora_v2"

TRAIN_SIZE = 20

MAX_STEPS = 10

NUM_GENERATIONS = 2

MAX_COMPLETION_LENGTH = 3000

TEMPERATURE = 0.7

LENGTH_SCALE = 1000.0


print("=" * 60)
print("Qwen3-1.7B QLoRA + GRPO v2")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 1. Clean model loading
#
# IMPORTANT:
# This cell should be run with a fresh model object.
# ============================================================

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

print("Loading 4bit model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto",

    dtype=torch.bfloat16,
)

model.config.use_cache = False

print("Model loaded.")

print(
    "Model device:",
    next(model.parameters()).device,
)


# ============================================================
# 2. Dataset
# ============================================================

print("\nLoading GSM8K...")

dataset = load_dataset(
    "openai/gsm8k",
    "main",
    split="train",
)

dataset = dataset.select(
    range(min(TRAIN_SIZE, len(dataset)))
)

print(
    "Training examples:",
    len(dataset),
)


# ============================================================
# 3. Extract GSM8K answer
# ============================================================

def extract_gsm8k_answer(answer):

    m = re.search(
        r"####\s*([-+]?\d+(?:\.\d+)?)",
        answer,
    )

    if m:
        return m.group(1).strip()

    nums = re.findall(
        r"[-+]?\d+(?:\.\d+)?",
        answer,
    )

    if nums:
        return nums[-1].strip()

    return None


# ============================================================
# 4. Prompt
# ============================================================

SYSTEM_PROMPT = """You are a careful math problem solver.

Solve the problem correctly.

Think only as much as necessary.
Avoid unnecessary reasoning and repetition.

However, do not skip reasoning that is genuinely necessary
to obtain the correct answer.

At the end, give the final numerical answer clearly.
"""


def make_prompt(example):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },

        {
            "role": "user",
            "content": (
                "Solve the following math problem.\n\n"
                + example["question"]
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return {
        "prompt": prompt,

        "answer": extract_gsm8k_answer(
            example["answer"]
        ),
    }


dataset = dataset.map(
    make_prompt,
    remove_columns=dataset.column_names,
)

print("\nExample:")

print(dataset[0])


# ============================================================
# 5. Model answer extraction
# ============================================================

def extract_model_answer(text):

    # #### 72
    matches = re.findall(
        r"####\s*([-+]?\d+(?:\.\d+)?)",
        text,
    )

    if matches:
        return matches[-1]

    # \boxed{72}
    matches = re.findall(
        r"\\boxed\{\s*([-+]?\d+(?:\.\d+)?)\s*\}",
        text,
    )

    if matches:
        return matches[-1]

    # answer is 72
    matches = re.findall(
        r"(?:final answer|answer)"
        r"\s*(?:is|:|=)\s*"
        r"([-+]?\d+(?:\.\d+)?)",

        text,

        flags=re.IGNORECASE,
    )

    if matches:
        return matches[-1]

    # Last numerical value
    matches = re.findall(
        r"[-+]?\d+(?:\.\d+)?",
        text,
    )

    if matches:
        return matches[-1]

    return None


# ============================================================
# 6. Number normalization
# ============================================================

def normalize_number(x):

    if x is None:
        return None

    try:

        value = float(x)

        if value.is_integer():
            return str(int(value))

        return str(value)

    except Exception:

        return str(x).strip()


# ============================================================
# 7. Token counting
# ============================================================

def count_tokens(text):

    ids = tokenizer(
        text,
        add_special_tokens=False,
    )["input_ids"]

    return len(ids)


# ============================================================
# 8. Convert TRL completion -> string
# ============================================================

def completion_to_text(completion):

    if isinstance(completion, str):

        return completion

    if isinstance(completion, list):

        parts = []

        for item in completion:

            if isinstance(item, dict):

                parts.append(
                    str(
                        item.get(
                            "content",
                            ""
                        )
                    )
                )

            else:

                parts.append(
                    str(item)
                )

        return "".join(parts)

    return str(completion)


# ============================================================
# 9. NEW REWARD
#
# Correctness dominates everything.
#
# Correct:
#
#     2.0 + brevity bonus
#
# Incorrect:
#
#     0.0
#
# Therefore:
#
#     correct > incorrect
#
# And among correct:
#
#     shorter > longer
#
# IMPORTANT:
# Incorrect answers receive NO length reward.
# ============================================================

def reward_function(
    completions,
    answer,
    **kwargs,
):

    rewards = []

    for completion, gold in zip(
        completions,
        answer,
    ):

        text = completion_to_text(
            completion
        )

        prediction = extract_model_answer(
            text
        )

        prediction = normalize_number(
            prediction
        )

        gold = normalize_number(
            gold
        )

        correct = (
            prediction is not None
            and gold is not None
            and prediction == gold
        )

        tokens = count_tokens(text)

        if correct:

            # ------------------------------------------------
            # Correct answer:
            #
            # 2.0 base reward
            #
            # + up to 1.0 for being short
            #
            # 0 tokens  -> ~3.0
            # 500 tokens -> ~2.5
            # 1000 tokens -> 2.0
            # 2000 tokens -> 2.0
            #
            # The floor at 2.0 means:
            # correctness remains dominant.
            # ------------------------------------------------

            brevity_bonus = max(
                0.0,
                1.0 - (
                    tokens / LENGTH_SCALE
                ),
            )

            reward = (
                2.0
                + brevity_bonus
            )

        else:

            # ------------------------------------------------
            # WRONG = WRONG.
            #
            # Do not reward longer wrong answers.
            # ------------------------------------------------

            reward = 0.0

        rewards.append(
            float(reward)
        )

    return rewards


# ============================================================
# 10. Reward sanity check
# ============================================================

print("\nReward sanity check:")

test_completions = [

    "The answer is 72.",

    (
        "We calculate this carefully. "
        "First we consider the quantities. "
        "The answer is 72."
    ),

    "The answer is 73.",

    (
        "Let me think carefully for a very long time. "
        "I will consider many possibilities. "
        "The answer is 73."
    ),
]

test_answers = [
    "72",
    "72",
    "72",
    "72",
]

test_rewards = reward_function(
    test_completions,
    test_answers,
)

for text, reward in zip(
    test_completions,
    test_rewards,
):

    print(
        f"tokens={count_tokens(text):4d} "
        f"reward={reward:.3f} | "
        f"{text[:100]}"
    )


# ============================================================
# 11. Prepare model for QLoRA
# ============================================================

print("\nPreparing model for QLoRA...")

model = prepare_model_for_kbit_training(
    model
)

model.config.use_cache = False


# ============================================================
# 12. LoRA configuration
# ============================================================

lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


# ============================================================
# 13. Verify trainable parameters
# ============================================================

# IMPORTANT:
# GRPOTrainer will attach the LoRA adapter.
# We check the base model BEFORE trainer creation.

print("\nBase model PEFT state:")

print(
    "Has peft_config:",
    hasattr(model, "peft_config")
)


# ============================================================
# 14. GRPO configuration
# ============================================================

training_args = GRPOConfig(

    output_dir=OUTPUT_DIR,

    # --------------------------------------------------------
    # A/B generation
    # --------------------------------------------------------

    num_generations=NUM_GENERATIONS,

    generation_batch_size=NUM_GENERATIONS,

    # --------------------------------------------------------
    # IMPORTANT:
    # Large safety limit.
    #
    # This should NOT become the learning objective.
    # --------------------------------------------------------

    max_completion_length=MAX_COMPLETION_LENGTH,

    temperature=TEMPERATURE,

    # --------------------------------------------------------
    # Batch
    # --------------------------------------------------------

    per_device_train_batch_size=1,

    gradient_accumulation_steps=1,

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    learning_rate=1e-5,

    num_train_epochs=1,

    max_steps=MAX_STEPS,

    warmup_ratio=0.1,

    # --------------------------------------------------------
    # Memory
    # --------------------------------------------------------

    gradient_checkpointing=True,

    bf16=True,

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_steps=1,

    save_steps=MAX_STEPS,

    report_to="none",

    # --------------------------------------------------------
    # Generation backend
    # --------------------------------------------------------

    use_vllm=False,
)


# ============================================================
# 15. Create trainer
# ============================================================

print("\nCreating GRPO trainer...")

trainer = GRPOTrainer(

    model=model,

    reward_funcs=reward_function,

    args=training_args,

    train_dataset=dataset,

    processing_class=tokenizer,

    peft_config=lora_config,
)


# ============================================================
# 16. Verify LoRA
# ============================================================

print("\n============================================================")
print("TRAINABLE PARAMETERS")
print("============================================================")

trainable = 0
total = 0

for name, parameter in model.named_parameters():

    total += parameter.numel()

    if parameter.requires_grad:
        trainable += parameter.numel()

print(
    f"Trainable: {trainable:,}"
)

print(
    f"Total:     {total:,}"
)

print(
    f"Ratio:     {100 * trainable / total:.4f}%"
)


# ============================================================
# 17. Train
# ============================================================

print("\n============================================================")
print("START GRPO v2")
print("============================================================")

start_time = time.time()

trainer.train()

elapsed = time.time() - start_time


# ============================================================
# 18. Save
# ============================================================

print("\n============================================================")
print("TRAINING COMPLETE")
print("============================================================")

print(
    "Training time:",
    round(
        elapsed / 60,
        2
    ),
    "minutes"
)


trainer.save_model(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

print(
    "Saved to:",
    OUTPUT_DIR
)


# ============================================================
# 19. VRAM
# ============================================================

print("\nVRAM:")

print(
    "Allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)

print(
    "Reserved:",
    round(
        torch.cuda.memory_reserved()
        / 1024**3,
        2
    ),
    "GB"
)


Qwen3-1.7B QLoRA + GRPO v2
PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4

Loading tokenizer...
Loading 4bit model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.
Model device: cuda:0

Loading GSM8K...
Training examples: 20


Map:   0%|          | 0/20 [00:00<?, ? examples/s]


Example:
{'answer': '72', 'prompt': '<|im_start|>system\nYou are a careful math problem solver.\n\nSolve the problem correctly.\n\nThink only as much as necessary.\nAvoid unnecessary reasoning and repetition.\n\nHowever, do not skip reasoning that is genuinely necessary\nto obtain the correct answer.\n\nAt the end, give the final numerical answer clearly.\n<|im_end|>\n<|im_start|>user\nSolve the following math problem.\n\nNatalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>\n<|im_start|>assistant\n'}

Reward sanity check:
tokens=   7 reward=2.993 | The answer is 72.
tokens=  18 reward=2.982 | We calculate this carefully. First we consider the quantities. The answer is 72.
tokens=   7 reward=0.000 | The answer is 73.
tokens=  23 reward=0.000 | Let me think carefully for a very long time. I will consider many possibilities. The answer is 73.

Preparing model for QLoRA...

Bas

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



TRAINABLE PARAMETERS
Trainable: 17,432,576
Total:     1,033,364,480
Ratio:     1.6870%

START GRPO v2


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,-0.380800
2,1.033100
3,-0.640900
4,0.772100
5,0.000000
6,0.000000
7,-0.674200


In [9]:
# ============================================================
# 20. Quick evaluation
# ============================================================

from peft import PeftModel
import torch

print("\nLoading trained adapter...")

# trainer.save_model() 後なら、現在の model をそのまま使える
model.eval()

def generate_answer(prompt, max_new_tokens=3000):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )


correct = 0
total_tokens = 0

for i, example in enumerate(dataset):

    text = generate_answer(
        example["prompt"]
    )

    prediction = normalize_number(
        extract_model_answer(text)
    )

    gold = normalize_number(
        example["answer"]
    )

    tokens = count_tokens(text)

    ok = prediction == gold

    correct += int(ok)
    total_tokens += tokens

    print(
        f"\n[{i+1}/{len(dataset)}]"
    )

    print(
        f"correct={ok} "
        f"tokens={tokens}"
    )

    print(text[:1000])

print("\n============================================================")
print("RESULT")
print("============================================================")

print(
    "Accuracy:",
    correct / len(dataset)
)

print(
    "Average tokens:",
    total_tokens / len(dataset)
)



Loading trained adapter...

[1/20]
correct=True tokens=494
<think>
Okay, let's see. Natalia sold clips to 48 of her friends in April. So, that's straightforward. April has 48 clips sold. Then in May, she sold half as many as she did in April. Hmm, half as many as April's sales. So, half of 48. Let me calculate that first.

So, half of 48 is 48 divided by 2. Let me do that. 48 divided by 2 is 24. So, in May, she sold 24 clips. Then the total number of clips sold in April and May would be 48 plus 24. Let me add those together. 48 + 24 is 72. So, the total is 72?

Wait, let me check again. April: 48. May: half as many as April, which is 48/2 = 24. Total is 48 + 24 = 72. Yeah, that seems right. But wait, the question says "half as many clips in May." Is that half of April's sales or half of the total? Wait, the wording is "she sold half as many clips in May." So, compared to April, May was half as many. So, if April was 48, then May was 48/2 = 24. So total is 48 + 24 = 72. Yeah, that's th

KeyboardInterrupt: 

In [ ]:
# ============================================================
# BASELINE vs TRAINED LoRA
# Same 20 GSM8K examples
# ============================================================

import re
import time
import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "Qwen/Qwen3-1.7B"
ADAPTER_DIR = "./qwen3_1.7b_grpo_qlora_v2"

N = 20
MAX_NEW_TOKENS = 2048

TEMPERATURE = 0.7


# ============================================================
# Dataset
# ============================================================

dataset = load_dataset(
    "openai/gsm8k",
    "main",
    split="train",
).select(range(N))


# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# Model loader
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)

base_model.config.use_cache = True


print("Loading trained adapter...")

trained_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

trained_model.eval()

print("Adapter loaded.")


# ============================================================
# Prompt
# ============================================================

SYSTEM_PROMPT = """You are a careful math problem solver.

Solve the problem correctly.

Think only as much as necessary.
Avoid unnecessary reasoning and repetition.

However, do not skip reasoning that is genuinely necessary
to obtain the correct answer.

At the end, give the final numerical answer clearly.
"""


def make_prompt(example):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                "Solve the following math problem.\n\n"
                + example["question"]
            ),
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


# ============================================================
# Gold answer
# ============================================================

def extract_gsm8k_answer(answer):

    m = re.search(
        r"####\s*([-+]?\d+(?:\.\d+)?)",
        answer,
    )

    if m:
        return m.group(1)

    return None


# ============================================================
# Model answer
# ============================================================

def extract_model_answer(text):

    matches = re.findall(
        r"####\s*([-+]?\d+(?:\.\d+)?)",
        text,
    )

    if matches:
        return matches[-1]

    matches = re.findall(
        r"\\boxed\{\s*([-+]?\d+(?:\.\d+)?)\s*\}",
        text,
    )

    if matches:
        return matches[-1]

    matches = re.findall(
        r"(?:final answer|answer)"
        r"\s*(?:is|:|=)\s*"
        r"([-+]?\d+(?:\.\d+)?)",
        text,
        flags=re.IGNORECASE,
    )

    if matches:
        return matches[-1]

    matches = re.findall(
        r"[-+]?\d+(?:\.\d+)?",
        text,
    )

    if matches:
        return matches[-1]

    return None


def normalize_number(x):

    if x is None:
        return None

    try:
        value = float(x)

        if value.is_integer():
            return str(int(value))

        return str(value)

    except:
        return str(x).strip()


# ============================================================
# Think extraction
# ============================================================

def split_think_final(text):

    m = re.search(
        r"<think>(.*?)</think>",
        text,
        flags=re.DOTALL,
    )

    if m:

        think = m.group(1).strip()

        final = (
            text[:m.start()]
            + text[m.end():]
        ).strip()

        return think, final

    return "", text.strip()


# ============================================================
# Generation
# ============================================================

def generate(model, prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    start = time.time()

    with torch.no_grad():

        outputs = model.generate(
            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=True,

            temperature=TEMPERATURE,

            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.time() - start

    generated = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return text, elapsed


# ============================================================
# Evaluation
# ============================================================

def evaluate(model, name):

    results = []

    print("\n")
    print("=" * 70)
    print(name)
    print("=" * 70)

    for i, example in enumerate(dataset):

        prompt = make_prompt(example)

        gold = normalize_number(
            extract_gsm8k_answer(
                example["answer"]
            )
        )

        text, elapsed = generate(
            model,
            prompt,
        )

        prediction = normalize_number(
            extract_model_answer(text)
        )

        correct = (
            prediction is not None
            and prediction == gold
        )

        think, final = split_think_final(
            text
        )

        total_tokens = len(
            tokenizer(
                text,
                add_special_tokens=False,
            )["input_ids"]
        )

        think_tokens = len(
            tokenizer(
                think,
                add_special_tokens=False,
            )["input_ids"]
        )

        final_tokens = len(
            tokenizer(
                final,
                add_special_tokens=False,
            )["input_ids"]
        )

        limit = (
            total_tokens >=
            MAX_NEW_TOKENS - 5
        )

        result = {
            "index": i,
            "correct": correct,
            "think": think_tokens,
            "final": final_tokens,
            "total": total_tokens,
            "latency": elapsed,
            "limit": limit,
            "text": text,
        }

        results.append(result)

        print(
            f"[{i+1:02d}/{N}] "
            f"correct={correct} "
            f"think={think_tokens:4d} "
            f"final={final_tokens:4d} "
            f"total={total_tokens:4d} "
            f"time={elapsed:6.2f}s "
            f"limit={limit}"
        )

    return results


# ============================================================
# Run
# ============================================================

base_results = evaluate(
    base_model,
    "BASE MODEL",
)

trained_results = evaluate(
    trained_model,
    "TRAINED LoRA",
)


# ============================================================
# Summary
# ============================================================

def summarize(results):

    accuracy = sum(
        r["correct"]
        for r in results
    ) / len(results)

    avg_think = sum(
        r["think"]
        for r in results
    ) / len(results)

    avg_final = sum(
        r["final"]
        for r in results
    ) / len(results)

    avg_total = sum(
        r["total"]
        for r in results
    ) / len(results)

    avg_latency = sum(
        r["latency"]
        for r in results
    ) / len(results)

    limit_count = sum(
        r["limit"]
        for r in results
    )

    return {
        "accuracy": accuracy,
        "think": avg_think,
        "final": avg_final,
        "total": avg_total,
        "latency": avg_latency,
        "limit": limit_count,
    }


b = summarize(base_results)
t = summarize(trained_results)


print("\n")
print("=" * 70)
print("BASELINE vs TRAINED")
print("=" * 70)

print(
    f"{'Metric':<20}"
    f"{'Base':>12}"
    f"{'Trained':>12}"
    f"{'Change':>12}"
)

print("-" * 70)

def row(name, x, y):

    if x != 0:
        change = (y - x) / x * 100
        change_text = f"{change:+.1f}%"
    else:
        change_text = "-"

    print(
        f"{name:<20}"
        f"{x:>12.1f}"
        f"{y:>12.1f}"
        f"{change_text:>12}"
    )


row(
    "Accuracy",
    b["accuracy"] * 100,
    t["accuracy"] * 100,
)

row(
    "Avg think tokens",
    b["think"],
    t["think"],
)

row(
    "Avg final tokens",
    b["final"],
    t["final"],
)

row(
    "Avg total tokens",
    b["total"],
    t["total"],
)

row(
    "Avg latency (sec)",
    b["latency"],
    t["latency"],
)

row(
    "Limit hit",
    b["limit"],
    t["limit"],
)


# ============================================================
# Per-example comparison
# ============================================================

print("\n")
print("=" * 70)
print("PER-EXAMPLE COMPARISON")
print("=" * 70)

for b_r, t_r in zip(
    base_results,
    trained_results,
):

    think_change = (
        t_r["think"] - b_r["think"]
    )

    total_change = (
        t_r["total"] - b_r["total"]
    )

    print(
        f"[{b_r['index']+1:02d}] "
        f"correct "
        f"{b_r['correct']} -> {t_r['correct']} | "
        f"think "
        f"{b_r['think']} -> {t_r['think']} "
        f"({think_change:+d}) | "
        f"total "
        f"{b_r['total']} -> {t_r['total']} "
        f"({total_change:+d})"
    )


Loading base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading trained adapter...
Adapter loaded.


BASE MODEL
